# 🎹 Notes to Keys — Training Notebook

This notebook trains an **Onsets and Frames** piano transcription model on the preprocessed MAESTRO dataset.

### Before you start:
1. ✅ **Enable GPU:** Runtime → Change runtime type → T4 GPU → Save
2. ✅ **Mount Google Drive** (Cell 1.2 below)
3. ✅ **Set your Drive path** in Cell 1.3

### Notebook Structure:
| Section | What it does |
|---------|-------------|
| 1. Setup | Mount Drive, install packages, verify GPU |
| 2. Config | All hyperparameters in one place |
| 3. Data | Dataset class, data loaders |
| 4. Model | Architecture definition |
| 5. Training | Training + validation loop |
| 6. Evaluation | Test set metrics |
| 7. Inference | Quick demo on a sample |

---
## Section 1: Setup

In [ ]:
# ── 1.1  Check GPU ──────────────────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 1.2  Mount Google Drive ──────────────────────────────────────────────────
# A browser popup will appear asking you to sign in — that is normal and expected.
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# ── 1.3  Set Your Paths ──────────────────────────────────────────────────────
# ⬇️  EDIT THESE to match where you stored files in Google Drive
import os

DRIVE_ROOT       = '/content/drive/MyDrive/notes_to_keys'
CHUNKS_DIR       = os.path.join(DRIVE_ROOT, 'processed_chunks')  # folder containing 2004/, 2006/ etc.
CHECKPOINT_DIR   = os.path.join(DRIVE_ROOT, 'checkpoints')
BEST_MODEL_PATH  = os.path.join(DRIVE_ROOT, 'best_model.pth')

# Create checkpoint dir if it doesn't exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Verify chunks directory exists
if os.path.exists(CHUNKS_DIR):
    year_folders = [d for d in os.listdir(CHUNKS_DIR) if os.path.isdir(os.path.join(CHUNKS_DIR, d))]
    print(f'✅ Found chunks directory: {CHUNKS_DIR}')
    print(f'   Year folders: {sorted(year_folders)}')
else:
    print(f'❌ Chunks directory not found: {CHUNKS_DIR}')
    print('   Please update the CHUNKS_DIR path above.')

In [ ]:
# ── 1.4  Install Required Packages ──────────────────────────────────────────
# Most are pre-installed on Colab — we just need mir_eval for evaluation
!pip install mir_eval --quiet

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.notebook import tqdm
import json
import time
import glob
import random
import matplotlib.pyplot as plt
import mir_eval

print('✅ All packages loaded')

---
## Section 2: Configuration

All hyperparameters are in one place. Edit this cell if you want to experiment.

In [ ]:
# ── 2.1  Training Configuration ─────────────────────────────────────────────

CFG = {
    # Data
    'chunk_dir':        CHUNKS_DIR,
    'val_years':        ['2006'],          # held out for validation
    'test_years':       ['2009'],          # held out for final test
    # (all other years = training)

    # Model
    'n_freq_bins':      176,               # CQT bins (must match preprocessing)
    'n_keys':           88,                # piano keys
    'chunk_frames':     100,               # time frames per chunk
    'cnn_channels':     [32, 64, 128],     # CNN feature map sizes
    'lstm_hidden':      256,               # LSTM hidden units (per direction)
    'lstm_layers':      2,                 # LSTM depth
    'dropout':          0.3,

    # Training
    'batch_size':       32,
    'lr':               3e-4,
    'weight_decay':     1e-4,
    'max_epochs':       50,
    'early_stop_patience': 5,
    'lr_patience':      3,
    'grad_clip':        1.0,
    'onset_loss_weight':2.0,               # weight onset more than frame
    'frame_loss_weight':1.0,

    # Evaluation
    'onset_threshold':  0.5,
    'frame_threshold':  0.5,

    # System
    'num_workers':      2,
    'seed':             42,
}

# Reproducibility
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])

print('Configuration set:')
for k, v in CFG.items():
    print(f'  {k:25s}: {v}')

---
## Section 3: Data Loading

In [ ]:
# ── 3.1  Dataset Class ───────────────────────────────────────────────────────

class MaestroChunkDataset(Dataset):
    """
    Lazily loads preprocessed .npz chunks from disk.
    Does NOT load everything into RAM — reads one file at a time.
    """
    def __init__(self, chunk_dir: str, years: list, shuffle: bool = True):
        """
        Args:
            chunk_dir: Root directory containing year subfolders
            years:     List of year folders to include, e.g. ['2004', '2008']
            shuffle:   Whether to shuffle file list at init
        """
        self.chunk_paths = []

        for year in years:
            year_dir = Path(chunk_dir) / year
            if year_dir.exists():
                paths = sorted(year_dir.glob('chunk_*.npz'))
                self.chunk_paths.extend(paths)
            else:
                print(f'⚠️  Year folder not found: {year_dir}')

        if shuffle:
            random.shuffle(self.chunk_paths)

        print(f'Dataset: {len(years)} year(s), {len(self.chunk_paths):,} chunks')

    def __len__(self):
        return len(self.chunk_paths)

    def __getitem__(self, idx):
        data = np.load(self.chunk_paths[idx])
        # CQT: (1, 176, 100) → keep as-is for CNN input
        cqt   = torch.tensor(data['cqt'],   dtype=torch.float32)   # (1, 176, 100)
        onset = torch.tensor(data['onset'], dtype=torch.float32)   # (100, 88)
        frame = torch.tensor(data['frame'], dtype=torch.float32)   # (100, 88)
        return cqt, onset, frame

In [ ]:
# ── 3.2  Build Dataloaders ───────────────────────────────────────────────────

all_years = ['2004', '2006', '2008', '2009', '2011', '2013', '2014', '2015', '2017', '2018']
val_years  = CFG['val_years']
test_years = CFG['test_years']
train_years = [y for y in all_years if y not in val_years and y not in test_years]

print(f'Train years: {train_years}')
print(f'Val years:   {val_years}')
print(f'Test years:  {test_years}')
print()

train_dataset = MaestroChunkDataset(CFG['chunk_dir'], train_years, shuffle=True)
val_dataset   = MaestroChunkDataset(CFG['chunk_dir'], val_years,   shuffle=False)
test_dataset  = MaestroChunkDataset(CFG['chunk_dir'], test_years,  shuffle=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
)

# Verify a batch
sample_cqt, sample_onset, sample_frame = next(iter(train_loader))
print(f'\nSample batch shapes:')
print(f'  CQT:   {sample_cqt.shape}')    # (32, 1, 176, 100)
print(f'  Onset: {sample_onset.shape}')  # (32, 100, 88)
print(f'  Frame: {sample_frame.shape}')  # (32, 100, 88)
print('✅ Data loaders ready')

In [ ]:
# ── 3.3  Quick Data Sanity Check ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(sample_cqt[0, 0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('CQT Input (sample 0)')
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Frequency bins')

axes[1].imshow(sample_onset[0].T.numpy(), aspect='auto', origin='lower', cmap='hot')
axes[1].set_title('Onset Roll (sample 0)')
axes[1].set_xlabel('Time frames')
axes[1].set_ylabel('Piano keys')

axes[2].imshow(sample_frame[0].T.numpy(), aspect='auto', origin='lower', cmap='hot')
axes[2].set_title('Frame Roll (sample 0)')
axes[2].set_xlabel('Time frames')
axes[2].set_ylabel('Piano keys')

plt.tight_layout()
plt.show()
print(f'CQT range: [{sample_cqt.min():.2f}, {sample_cqt.max():.2f}]')
print(f'Onset active: {sample_onset.mean():.4f} ({sample_onset.mean()*100:.2f}%)')
print(f'Frame active: {sample_frame.mean():.4f} ({sample_frame.mean()*100:.2f}%)')

---
## Section 4: Model Architecture

In [ ]:
# ── 4.1  Acoustic CNN ────────────────────────────────────────────────────────

class AcousticCNN(nn.Module):
    """
    Shared CNN backbone. Extracts acoustic features from CQT input.

    Input:  (batch, 1, 176, 100)  — (batch, channels, freq_bins, time_frames)
    Output: (batch, 100, feature_dim)  — one feature vector per time frame
    """
    def __init__(self, n_freq_bins=176, channels=[32, 64, 128], dropout=0.3):
        super().__init__()

        self.conv_layers = nn.Sequential(
            # Block 1
            nn.Conv2d(1, channels[0], kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(),
            nn.Dropout2d(dropout * 0.5),

            # Block 2
            nn.Conv2d(channels[0], channels[1], kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(channels[1]),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve freq, keep time
            nn.Dropout2d(dropout * 0.5),

            # Block 3
            nn.Conv2d(channels[1], channels[2], kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(channels[2]),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve freq again, keep time
            nn.Dropout2d(dropout),
        )

        # Calculate output feature dim after pooling
        # Two MaxPool2d(2,1) on freq: 176 → 88 → 44
        self.freq_after_pool = n_freq_bins // 4
        self.feature_dim = channels[2] * self.freq_after_pool

    def forward(self, x):
        # x: (batch, 1, 176, 100)
        x = self.conv_layers(x)              # (batch, 128, 44, 100)
        batch, c, f, t = x.shape
        x = x.view(batch, c * f, t)         # (batch, 128*44, 100)
        x = x.permute(0, 2, 1)              # (batch, 100, feature_dim)
        return x

In [ ]:
# ── 4.2  Full Onsets and Frames Model ────────────────────────────────────────

class OnsetsAndFrames(nn.Module):
    """
    Piano transcription model with separate onset and frame detectors.

    Input:  CQT (batch, 1, 176, 100)
    Output: onset_logits (batch, 100, 88)
            frame_logits (batch, 100, 88)
    """
    def __init__(self, cfg: dict):
        super().__init__()

        # Shared CNN backbone
        self.cnn = AcousticCNN(
            n_freq_bins=cfg['n_freq_bins'],
            channels=cfg['cnn_channels'],
            dropout=cfg['dropout']
        )
        feat_dim = self.cnn.feature_dim
        hidden   = cfg['lstm_hidden']
        layers   = cfg['lstm_layers']
        n_keys   = cfg['n_keys']

        # Onset detector
        self.onset_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg['dropout'] if layers > 1 else 0.0,
        )
        self.onset_proj = nn.Sequential(
            nn.Dropout(cfg['dropout']),
            nn.Linear(hidden * 2, n_keys),   # *2 for bidirectional
        )

        # Frame detector (conditioned on onset)
        # Input = CNN features + onset predictions
        self.frame_lstm = nn.LSTM(
            input_size=feat_dim + n_keys,    # concat CNN features + onset preds
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg['dropout'] if layers > 1 else 0.0,
        )
        self.frame_proj = nn.Sequential(
            nn.Dropout(cfg['dropout']),
            nn.Linear(hidden * 2, n_keys),
        )

    def forward(self, cqt):
        # 1. Extract acoustic features
        features = self.cnn(cqt)                          # (B, T, feat_dim)

        # 2. Onset branch
        onset_out, _ = self.onset_lstm(features)          # (B, T, hidden*2)
        onset_logits = self.onset_proj(onset_out)         # (B, T, 88)

        # 3. Frame branch — conditioned on onset predictions
        onset_probs = torch.sigmoid(onset_logits).detach()  # stop gradient flow
        frame_input = torch.cat([features, onset_probs], dim=-1)  # (B, T, feat+88)
        frame_out, _ = self.frame_lstm(frame_input)        # (B, T, hidden*2)
        frame_logits = self.frame_proj(frame_out)          # (B, T, 88)

        return onset_logits, frame_logits

In [ ]:
# ── 4.3  Instantiate and Inspect ─────────────────────────────────────────────

model = OnsetsAndFrames(CFG).to(device)

# Count parameters
total_params   = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model size (approx):  {total_params * 4 / 1e6:.1f} MB')

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 1, 176, 100).to(device)
    onset_out, frame_out = model(dummy)
    print(f'\nForward pass check:')
    print(f'  Onset logits: {onset_out.shape}')   # (2, 100, 88)
    print(f'  Frame logits: {frame_out.shape}')   # (2, 100, 88)
print('✅ Model ready')

---
## Section 5: Training

In [ ]:
# ── 5.1  Loss and Optimizer ──────────────────────────────────────────────────

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay']
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=CFG['lr_patience'],
    factor=0.5,
    verbose=True
)

print('Optimizer:', optimizer)
print('Loss: BCEWithLogitsLoss')

In [ ]:
# ── 5.2  Checkpoint Utilities ────────────────────────────────────────────────

def save_checkpoint(epoch, model, optimizer, val_loss, path):
    torch.save({
        'epoch':      epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss':   val_loss,
        'cfg':        CFG,
    }, path)

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch    = checkpoint['epoch']
    val_loss = checkpoint['val_loss']
    print(f'Loaded checkpoint: epoch {epoch}, val_loss {val_loss:.4f}')
    return epoch, val_loss

# ── Optionally resume from a checkpoint ─────────────────────────────────────
RESUME_FROM = None  # set to a path like BEST_MODEL_PATH to resume

start_epoch = 0
best_val_loss = float('inf')

if RESUME_FROM and os.path.exists(RESUME_FROM):
    start_epoch, best_val_loss = load_checkpoint(RESUME_FROM, model, optimizer)
    start_epoch += 1
    print(f'Resuming from epoch {start_epoch}')
else:
    print('Starting fresh training')

In [ ]:
# ── 5.3  Training and Validation Step Functions ───────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for cqt, onset_target, frame_target in tqdm(loader, desc='  Train', leave=False):
        cqt          = cqt.to(device)
        onset_target = onset_target.to(device)
        frame_target = frame_target.to(device)

        optimizer.zero_grad()

        onset_logits, frame_logits = model(cqt)

        onset_loss = criterion(onset_logits, onset_target)
        frame_loss = criterion(frame_logits, frame_target)
        loss = CFG['onset_loss_weight'] * onset_loss + CFG['frame_loss_weight'] * frame_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    all_onset_preds  = []
    all_onset_targets = []
    all_frame_preds  = []
    all_frame_targets = []

    for cqt, onset_target, frame_target in tqdm(loader, desc='  Val  ', leave=False):
        cqt          = cqt.to(device)
        onset_target = onset_target.to(device)
        frame_target = frame_target.to(device)

        onset_logits, frame_logits = model(cqt)

        onset_loss = criterion(onset_logits, onset_target)
        frame_loss = criterion(frame_logits, frame_target)
        loss = CFG['onset_loss_weight'] * onset_loss + CFG['frame_loss_weight'] * frame_loss

        total_loss += loss.item()
        n_batches  += 1

        # Accumulate predictions for F1 metric
        onset_preds = (torch.sigmoid(onset_logits) > CFG['onset_threshold']).float()
        frame_preds = (torch.sigmoid(frame_logits) > CFG['frame_threshold']).float()

        all_onset_preds.append(onset_preds.cpu())
        all_onset_targets.append(onset_target.cpu())
        all_frame_preds.append(frame_preds.cpu())
        all_frame_targets.append(frame_target.cpu())

    # Compute frame-level F1
    onset_preds_cat  = torch.cat(all_onset_preds).numpy().flatten()
    onset_targets_cat = torch.cat(all_onset_targets).numpy().flatten()
    frame_preds_cat  = torch.cat(all_frame_preds).numpy().flatten()
    frame_targets_cat = torch.cat(all_frame_targets).numpy().flatten()

    def f1(pred, target):
        tp = (pred * target).sum()
        fp = (pred * (1 - target)).sum()
        fn = ((1 - pred) * target).sum()
        precision = tp / (tp + fp + 1e-8)
        recall    = tp / (tp + fn + 1e-8)
        return 2 * precision * recall / (precision + recall + 1e-8)

    onset_f1 = f1(onset_preds_cat, onset_targets_cat)
    frame_f1 = f1(frame_preds_cat, frame_targets_cat)

    return total_loss / n_batches, float(onset_f1), float(frame_f1)

In [ ]:
# ── 5.4  Main Training Loop ───────────────────────────────────────────────────

history = {'train_loss': [], 'val_loss': [], 'onset_f1': [], 'frame_f1': []}
epochs_no_improve = 0

print('='*65)
print('Starting training...')
print(f'Train batches/epoch: {len(train_loader):,}')
print(f'Val   batches/epoch: {len(val_loader):,}')
print('='*65)

for epoch in range(start_epoch, CFG['max_epochs']):
    epoch_start = time.time()

    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, device, CFG['grad_clip']
    )

    # Validate
    val_loss, onset_f1, frame_f1 = validate_epoch(
        model, val_loader, criterion, device
    )

    # LR scheduler step
    scheduler.step(val_loss)

    # Log
    elapsed = time.time() - epoch_start
    lr_now  = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['onset_f1'].append(onset_f1)
    history['frame_f1'].append(frame_f1)

    print(f'Epoch {epoch+1:3d}/{CFG["max_epochs"]} | '
          f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
          f'Onset F1: {onset_f1:.3f} | Frame F1: {frame_f1:.3f} | '
          f'LR: {lr_now:.2e} | {elapsed:.0f}s')

    # Save per-epoch checkpoint to Drive
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1:03d}.pth')
    save_checkpoint(epoch, model, optimizer, val_loss, ckpt_path)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(epoch, model, optimizer, val_loss, BEST_MODEL_PATH)
        print(f'  ✅ New best model saved (val_loss={val_loss:.4f})')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # Early stopping
    if epochs_no_improve >= CFG['early_stop_patience']:
        print(f'\n⏹️  Early stopping: no improvement for {CFG["early_stop_patience"]} epochs')
        break

print('\n' + '='*65)
print(f'Training complete. Best val loss: {best_val_loss:.4f}')
print(f'Best model saved at: {BEST_MODEL_PATH}')

In [ ]:
# ── 5.5  Training Curves ──────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_ran, history['train_loss'], label='Train loss', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val loss',   linewidth=2)
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['onset_f1'], label='Onset F1', linewidth=2)
axes[1].plot(epochs_ran, history['frame_f1'], label='Frame F1', linewidth=2)
axes[1].set_title('Validation F1 Scores')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved to Drive')

---
## Section 6: Evaluation on Test Set

In [ ]:
# ── 6.1  Load Best Model and Evaluate ────────────────────────────────────────

# Load the best checkpoint
load_checkpoint(BEST_MODEL_PATH, model)
model.eval()

test_loss, test_onset_f1, test_frame_f1 = validate_epoch(
    model, test_loader, criterion, device
)

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
print(f'Test Loss:    {test_loss:.4f}')
print(f'Onset F1:     {test_onset_f1:.4f}')
print(f'Frame F1:     {test_frame_f1:.4f}')
print('='*50)

if test_frame_f1 > 0.80:
    print('🎉 Excellent! Frame F1 > 0.80 — publication-level performance!')
elif test_frame_f1 > 0.70:
    print('✅ Good performance. Consider more epochs or hyperparameter tuning.')
else:
    print('⚠️  Performance is lower than expected. Review training curves for issues.')

In [ ]:
# ── 6.2  Visualize a Test Prediction ─────────────────────────────────────────

model.eval()
with torch.no_grad():
    test_cqt, test_onset_gt, test_frame_gt = next(iter(test_loader))
    onset_logits, frame_logits = model(test_cqt.to(device))

sample_idx = 0
onset_pred = (torch.sigmoid(onset_logits[sample_idx]) > CFG['onset_threshold']).cpu().numpy()
frame_pred = (torch.sigmoid(frame_logits[sample_idx]) > CFG['frame_threshold']).cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
titles = ['CQT Input', 'Onset (Ground Truth)', 'Frame (Ground Truth)',
          '', 'Onset (Predicted)', 'Frame (Predicted)']
data   = [
    test_cqt[sample_idx, 0].numpy(),
    test_onset_gt[sample_idx].T.numpy(),
    test_frame_gt[sample_idx].T.numpy(),
    None,
    onset_pred.T,
    frame_pred.T,
]

for i, (ax, title, d) in enumerate(zip(axes.flat, titles, data)):
    if d is None:
        ax.axis('off')
        continue
    cmap = 'viridis' if i == 0 else 'hot'
    ax.imshow(d, aspect='auto', origin='lower', cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Piano keys')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'test_prediction_sample.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7: Quick Inference Demo

Run inference on a single audio file — this is the same pipeline the Flask backend will use.

In [ ]:
# ── 7.1  Single-File Inference Function ──────────────────────────────────────
# Install librosa if not already available
try:
    import librosa
except ImportError:
    !pip install librosa --quiet
    import librosa

def preprocess_audio(wav_path: str, cfg: dict) -> torch.Tensor:
    """Convert a WAV file to CQT chunks ready for the model."""
    signal, sr = librosa.load(wav_path, sr=22050)
    cqt = librosa.cqt(
        signal, sr=sr, fmin=27.5, n_bins=176,
        bins_per_octave=24, hop_length=512
    )
    cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
    cqt_norm = (cqt_db - (-40.0)) / 20.0

    # Split into chunks of 100 frames
    chunk_len = cfg['chunk_frames']
    n_frames = cqt_norm.shape[1]
    chunks = []
    for start in range(0, n_frames - chunk_len + 1, chunk_len):
        chunk = cqt_norm[:, start:start + chunk_len]
        chunks.append(chunk[np.newaxis, :, :])  # (1, 176, 100)

    return torch.tensor(np.array(chunks), dtype=torch.float32)  # (N, 1, 176, 100)


def transcribe_audio(wav_path: str, model, device, cfg: dict) -> dict:
    """Full inference pipeline: audio → piano roll."""
    model.eval()
    chunks = preprocess_audio(wav_path, cfg).to(device)  # (N, 1, 176, 100)

    all_onset, all_frame = [], []
    with torch.no_grad():
        for i in range(0, len(chunks), 8):  # process 8 chunks at a time
            batch = chunks[i:i+8]
            onset_l, frame_l = model(batch)
            all_onset.append(torch.sigmoid(onset_l).cpu())
            all_frame.append(torch.sigmoid(frame_l).cpu())

    onset_roll = torch.cat(all_onset, dim=0).numpy()  # (N, 100, 88)
    frame_roll = torch.cat(all_frame, dim=0).numpy()  # (N, 100, 88)

    # Flatten chunks → single timeline
    onset_flat = onset_roll.reshape(-1, 88)           # (N*100, 88)
    frame_flat = frame_roll.reshape(-1, 88)

    return {
        'onset_roll': onset_flat,
        'frame_roll': frame_flat,
        'frame_duration_sec': 512 / 22050,
    }

print('Inference function defined')
print('Usage: result = transcribe_audio("path/to/file.wav", model, device, CFG)')

In [ ]:
# ── 7.2  Run on a Sample File (optional) ─────────────────────────────────────
# Upload a .wav file to Colab and set the path here

# from google.colab import files
# uploaded = files.upload()   # opens a file picker
# wav_file = list(uploaded.keys())[0]

# result = transcribe_audio(wav_file, model, device, CFG)
# print(f'Transcribed {result["onset_roll"].shape[0]} frames')
# print(f'Duration: {result["onset_roll"].shape[0] * result["frame_duration_sec"]:.1f} seconds')

print('Uncomment the block above and upload a .wav file to test inference.')

---
## Section 8: Download the Model

After training is complete, download `best_model.pth` to your local machine.

In [ ]:
# ── 8.1  Download Best Model ─────────────────────────────────────────────────
# The model is already saved to your Google Drive at BEST_MODEL_PATH.
# To also download it directly to your computer:

# from google.colab import files
# files.download(BEST_MODEL_PATH)

print(f'Best model location on Drive: {BEST_MODEL_PATH}')
print(f'Checkpoints location: {CHECKPOINT_DIR}')
print('\nTo download to your computer, uncomment the lines above.')